# Tests, Refactor KPConv Model

In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import numpy as np
from typing import List, Tuple, Any, Optional, Dict, Union
import random
import math
import torch
import torch.nn as nn

In [5]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## Utils

Refactor some utils like the spherical lloyd algorithm

In [6]:
# Original
import sys
sys.path.append('/home/arthur/Documents/Code/Github/KPConv-PyTorch/')


from kernels.kernel_points import (
    create_3D_rotations,
    spherical_Lloyd, 
    kernel_point_optimization_debug,
    load_kernels,
)

### Refactor initialization of kernel points

#### spherical Lloyd

In [10]:
# Original spherical Lloyd
set_seed(42)

radius = 1.0
num_points = 10
dimension = 3
fixed = 'center'
approximation = 'monte-carlo'
approx_n = 5000
max_iter = 500
momentum = 0.9
verbose = 0

# Radius used for optimization (points are rescaled afterwards)
radius0 = 1.0

# Random kernel points (Uniform distribution in a sphere)
kernel_points = np.zeros((0, dimension))
while kernel_points.shape[0] < num_points:
    new_points = np.random.rand(num_points, dimension) * 2 * radius0 - radius0
    kernel_points = np.vstack((kernel_points, new_points))
    d2 = np.sum(np.power(kernel_points, 2), axis=1)
    kernel_points = kernel_points[np.logical_and(d2 < radius0 ** 2, (0.9 * radius0) ** 2 < d2), :]
kernel_points = kernel_points[:num_points]

print(f"{kernel_points = }")

kernel_points = array([[ 0.19731697, -0.68796272, -0.68801096],
       [ 0.02846888,  0.18482914, -0.90709917],
       [-0.39077246, -0.80465577,  0.36846605],
       [ 0.24659625, -0.33820395, -0.8728833 ],
       [ 0.04546566, -0.14491796, -0.94916175],
       [ 0.26680751,  0.74292118,  0.60734415],
       [-0.77989615, -0.54412967, -0.14578442],
       [ 0.67060499, -0.35843987, -0.62696298],
       [ 0.75467871, -0.48411674,  0.31996809],
       [ 0.80083611,  0.26620291, -0.32194042]])


In [11]:
def random_spherical_points_numpy(
    num_points: int,
    radius: float = 1.0,
    ratio: Union[float, Tuple[float, float]] = 1.0,
    limits: Optional[Tuple[float, float]] = None,
) -> np.ndarray:
    if isinstance(ratio, float):
        ratio = (0.0, ratio)

    inner_limit, outer_limit = ratio
    
    points= np.zeros((0, 3))
    while points.shape[0] < num_points:
        # Generate random points in the bounding cube
        new_points = np.random.rand(num_points, 3) * 2 * radius - radius
        d2 = np.sum(np.power(new_points, 2), axis=1)

        # Filter points that fall within the spherical shell or full sphere as per the given range
        valid_points = new_points[np.logical_and(d2 < (outer_limit * radius) ** 2, d2 > (inner_limit * radius) ** 2)]
        points = np.vstack((points, valid_points))

    points = points[:num_points]
    
    return points


set_seed(42)
kernel_points_v2 = random_spherical_points_numpy(
    num_points=num_points, 
    radius=radius0, 
    ratio=(0.9, 1.0), 
)

print(f"{kernel_points_v2 = }")

kernel_points_v2 = array([[ 0.19731697, -0.68796272, -0.68801096],
       [ 0.02846888,  0.18482914, -0.90709917],
       [-0.39077246, -0.80465577,  0.36846605],
       [ 0.24659625, -0.33820395, -0.8728833 ],
       [ 0.04546566, -0.14491796, -0.94916175],
       [ 0.26680751,  0.74292118,  0.60734415],
       [-0.77989615, -0.54412967, -0.14578442],
       [ 0.67060499, -0.35843987, -0.62696298],
       [ 0.75467871, -0.48411674,  0.31996809],
       [ 0.80083611,  0.26620291, -0.32194042]])


In [12]:
np.allclose(kernel_points, kernel_points_v2)

True

#### For kernel points optimization debug

In [13]:
# Original
set_seed(42)

radius = 1.0
num_points = 10
dimension = 3
num_kernels = 1
fixed = 'center'
ratio = 0.66
verbose = 0

radius0 = 1.0
diameter0 = 2.0
moving_factor = 1e-2
continuous_moving_decay = 0.9995
thresh = 1e-5
clip = 0.05 * radius0

# Initialize random kernel points
kernel_points = np.random.rand(num_kernels * num_points - 1, dimension) * diameter0 - radius0
while kernel_points.shape[0] < num_kernels * num_points:
    new_points = np.random.rand(num_kernels * num_points - 1, dimension) * diameter0 - radius0
    kernel_points = np.vstack((kernel_points, new_points))
    d2 = np.sum(np.power(kernel_points, 2), axis=1)
    kernel_points = kernel_points[d2 < 0.5 * radius0 * radius0, :]
kernel_points = kernel_points[:num_kernels * num_points, :]

print(f"{kernel_points = }")

kernel_points = array([[-0.13610996, -0.41754172,  0.22370579],
       [ 0.32504457, -0.37657785,  0.04013604],
       [-0.28649335, -0.43813098,  0.08539217],
       [-0.37803536, -0.34963336,  0.45921236],
       [ 0.1225544 ,  0.54193436, -0.01240881],
       [ 0.02149461, -0.16517799, -0.55578438],
       [-0.35359414,  0.03758124,  0.40603792],
       [-0.49643541, -0.00550299, -0.39824338],
       [ 0.4564327 , -0.26443373,  0.26461166],
       [ 0.6344444 ,  0.11040162,  0.05930116]])


In [17]:
set_seed(42)
kernel_points_v2 = random_spherical_points_numpy(
    num_points=num_points, 
    radius=radius0, 
    ratio=(0, math.sqrt(0.5)), 
)

print(f"{kernel_points_v2 = }")

kernel_points_v2 = array([[-0.13610996, -0.41754172,  0.22370579],
       [ 0.32504457, -0.37657785,  0.04013604],
       [-0.28649335, -0.43813098,  0.08539217],
       [-0.37803536, -0.34963336,  0.45921236],
       [ 0.1225544 ,  0.54193436, -0.01240881],
       [ 0.02149461, -0.16517799, -0.55578438],
       [-0.35359414,  0.03758124,  0.40603792],
       [-0.49643541, -0.00550299, -0.39824338],
       [ 0.4564327 , -0.26443373,  0.26461166],
       [ 0.6344444 ,  0.11040162,  0.05930116]])


In [18]:
np.allclose(kernel_points, kernel_points_v2)

True

In [22]:
from torch_pointcloud.utils.plotly import plot_points


points = random_spherical_points_numpy(
    num_points=1000, 
    radius=1, 
    ratio=(0.9, 1.0),
)


plot_points(points)

### Kernel point optimization

In [267]:
def kernel_point_optimization(
    radius: float,
    num_points: int,
    num_kernels: int = 1,
    dimension: int = 3,
    fixed: str = 'center',
    ratio: float = 0.66
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Creation of kernel points via optimization of potentials.

    Args:
        radius: Radius of the kernels.
        num_points: Number of points composing the kernels.
        num_kernels: Number of kernels to generate.
        dimension: Dimension of the space.
        fixed: Fix position of certain kernel points ('none', 'center', or 'verticals').
        ratio: Ratio of the radius where you want the kernel points to be placed.

    Returns:
        A tuple containing:
            - Optimized kernel points of shape [num_kernels, num_points, dimension].
            - Saved gradient norms of the optimization process.
    """
    
    # Optimization parameters
    radius0 = 1.0
    diameter0 = 2.0
    moving_factor = 1e-2
    continuous_moving_decay = 0.9995
    thresh = 1e-5
    clip = 0.05 * radius0

    # Initialize random kernel points
    kernel_points = np.random.rand(num_kernels * num_points - 1, dimension) * diameter0 - radius0
    while kernel_points.shape[0] < num_kernels * num_points:
        new_points = np.random.rand(num_kernels * num_points - 1, dimension) * diameter0 - radius0
        kernel_points = np.vstack((kernel_points, new_points))
        d2 = np.sum(np.power(kernel_points, 2), axis=1)
        kernel_points = kernel_points[d2 < 0.5 * radius0 * radius0, :]
    kernel_points = kernel_points[:num_kernels * num_points, :].reshape((num_kernels, num_points, -1))

    # Fix certain kernel points if specified
    if fixed == 'center':
        kernel_points[:, 0, :] *= 0
    if fixed == 'verticals':
        kernel_points[:, :3, :] *= 0
        kernel_points[:, 1, -1] += 2 * radius0 / 3
        kernel_points[:, 2, -1] -= 2 * radius0 / 3

    # Kernel optimization
    saved_gradient_norms = np.zeros((10000, num_kernels))
    old_gradient_norms = np.zeros((num_kernels, num_points))
    step = -1

    while step < 10000:
        step += 1

        # Compute gradients
        A = np.expand_dims(kernel_points, axis=2)
        B = np.expand_dims(kernel_points, axis=1)
        interd2 = np.sum(np.power(A - B, 2), axis=-1)
        inter_grads = (A - B) / (np.power(np.expand_dims(interd2, -1), 3 / 2) + 1e-6)
        inter_grads = np.sum(inter_grads, axis=1)

        circle_grads = 10 * kernel_points

        # All gradients
        gradients = inter_grads + circle_grads

        if fixed == 'verticals':
            gradients[:, 1:3, :-1] = 0

        # Compute norm of gradients
        gradients_norms = np.sqrt(np.sum(np.power(gradients, 2), axis=-1))
        saved_gradient_norms[step, :] = np.max(gradients_norms, axis=1)

        # Stop condition
        if fixed == 'center' and np.max(np.abs(old_gradient_norms[:, 1:] - gradients_norms[:, 1:])) < thresh:
            break
        elif fixed == 'verticals' and np.max(np.abs(old_gradient_norms[:, 3:] - gradients_norms[:, 3:])) < thresh:
            break
        elif np.max(np.abs(old_gradient_norms - gradients_norms)) < thresh:
            break

        old_gradient_norms = gradients_norms

        # Move points
        moving_dists = np.minimum(moving_factor * gradients_norms, clip)

        if fixed == 'center':
            moving_dists[:, 0] = 0
        if fixed == 'verticals':
            moving_dists[:, 0] = 0

        kernel_points -= np.expand_dims(moving_dists, -1) * gradients / np.expand_dims(gradients_norms + 1e-6, -1)

        # Moving factor decay
        moving_factor *= continuous_moving_decay

    # Remove unused lines in the saved gradients
    if step < 10000:
        saved_gradient_norms = saved_gradient_norms[:step + 1, :]

    # Rescale radius to fit the wanted ratio of radius
    r = np.sqrt(np.sum(np.power(kernel_points, 2), axis=-1))
    kernel_points *= ratio / np.mean(r[:, 1:])

    # Rescale kernels with real radius
    return kernel_points * radius, saved_gradient_norms


set_seed(42)
kernel_points, saved_gradient_norms = kernel_point_optimization(
    radius=1.0,
    num_points=10,
    num_kernels=1,
    dimension=3,
    fixed='center',
    ratio=0.66,
)

print(f"{kernel_points = }")

kernel_points = array([[[ 0.        ,  0.        ,  0.        ],
        [ 0.37299413, -0.54306198, -0.00129056],
        [-0.31443067, -0.50028142, -0.2990839 ],
        [-0.29606397, -0.39011744,  0.44081876],
        [ 0.06703513,  0.64405153, -0.12285586],
        [ 0.07856441,  0.03180909, -0.65334399],
        [-0.31033923,  0.36197091,  0.45925462],
        [-0.59284425,  0.18934779, -0.21644799],
        [ 0.36806846,  0.05628052,  0.54382688],
        [ 0.62699197,  0.14985694, -0.15099412]]])


In [252]:
plot_points(kernel_points[0])

In [275]:
# TODO: add type hints for return_grads (bool)) which returns the gradient norms if True
def gradient_optimization_spherical_points(
    radius: float,
    num_points: int,
    fixed: str = 'center',
    ratio: float = 0.66,
    max_steps: int = 10_000,
    step_size: float = 1e-2,
    step_decay: float = 0.9995,
    convergence_threshold: float = 1e-5,
    max_step_size: Optional[float] = None,
) -> Tuple[np.ndarray, np.ndarray]:
    """Creation of kernel points via optimization of potentials for a single kernel.

    Args:
        radius: Radius of the kernel.
        num_points: Number of points composing the kernel.
        fixed: Fix position of certain kernel points ('none', 'center', or 'verticals').
        ratio: Ratio of the radius where you want the kernel points to be placed.
        max_steps: Maximum number of optimization steps.
        step_size: Step size for moving points based on gradient norms.
        step_decay: Decay factor for reducing the step size over time.
        convergence_threshold: Threshold for stopping the optimization when gradient norm changes are small.
        max_step_size: Maximum distance a point can move in a single step.

    Returns:
        A tuple containing:
            - Optimized kernel points of shape [num_points, dimension].
            - Saved gradient norms of the optimization process.
    """
    def compute_gradients(points: np.ndarray) -> np.ndarray:
        A = np.expand_dims(points, axis=1)
        B = np.expand_dims(points, axis=0)
        interd2 = np.sum(np.power(A - B, 2), axis=-1)
        inter_grads = (A - B) / (np.power(np.expand_dims(interd2, -1), 3 / 2) + 1e-6)
        inter_grads = np.sum(inter_grads, axis=0)
        circle_grads = 10 * points

        return inter_grads + circle_grads
    
    # Parameters
    if max_step_size is None:
        max_step_size = 0.05 * radius

    # Initialize kernel points
    kernel_points = random_spherical_points_numpy(num_points, radius, ratio=(0, 0.7071067811865476))

    # Apply fixed positions if required
    if fixed == 'center':
        kernel_points[0, :] = 0  # Fix the first point to the center
    elif fixed == 'verticals':
        kernel_points[:3, :] = 0  # Fix the first three points
        kernel_points[1, -1] += 2 * radius / 3  # Move second point up vertically
        kernel_points[2, -1] -= 2 * radius / 3  # Move third point down vertically

    # Kernel optimization
    saved_grad_norms = np.zeros((max_steps,))
    old_grad_norms = np.zeros((num_points,))
    step = 0

    while step < max_steps:
        grads = compute_gradients(kernel_points)
        
        if fixed == 'verticals':
            grads[1:3, :-1] = 0
        
        grad_norms = np.sqrt(np.sum(np.power(grads, 2), axis=-1))
        saved_grad_norms[step] = np.max(grad_norms)

        # Check for stopping conditions
        if fixed == 'center' and np.max(np.abs(old_grad_norms[1:] - grad_norms[1:])) < convergence_threshold:
            break
        elif fixed == 'verticals' and np.max(np.abs(old_grad_norms[3:] - grad_norms[3:])) < convergence_threshold:
            break
        elif np.max(np.abs(old_grad_norms - grad_norms)) < convergence_threshold:
            break

        old_grad_norms = grad_norms

        # Move points
        moving_dists = np.minimum(step_size * grad_norms, max_step_size)
        if fixed == 'center' or fixed == 'verticals':
            moving_dists[0] = 0  # Do not move the first point if fixed

        kernel_points -= np.expand_dims(moving_dists, -1) * grads / np.expand_dims(grad_norms + 1e-6, -1)
        step_size *= step_decay
        step += 1

    # Rescale kernel points
    r = np.sqrt(np.sum(np.power(kernel_points, 2), axis=-1))
    kernel_points *= ratio / np.mean(r[1:])

    return kernel_points, saved_grad_norms[step - 1]


set_seed(42)
kernel_points_v2, grad = gradient_optimization_spherical_points(
    radius=1.0,
    num_points=10,
    fixed='center',
    ratio=0.66,
)

In [276]:
np.allclose(kernel_points, kernel_points_v2)

True

In [16]:
from torch_pointcloud.utils.geometry import gradient_optimization_spherical_points, spherical_points_lloyd
from torch_pointcloud.utils.plotly import plot_points

kernel_points_v2, grad = gradient_optimization_spherical_points(
    radius=1.0,
    num_points=100,
    fixed='none',
    ratio=0.66,
)

plot_points(kernel_points_v2)

In [20]:
from torch_pointcloud.utils.geometry import gradient_optimization_spherical_points, spherical_points_lloyd

kernel_points = spherical_points_lloyd(
    num_points=100,
    radius=1.0,
    position='center',
)

plot_points(kernel_points)


### Load Kernels

In [51]:
# Refactored
from torch_pointcloud.models.kpconv import kernel_points

kernel = kernel_points(radius=1.0, num_points=100, fixed_position='center', algorithm='gradient')

In [52]:
plot_points(kernel)

---

# KPConv

In [194]:
from models.blocks import KPConv
from kernels.kernel_points import load_kernels


set_seed(42)

# Define the model
model = KPConv(
    kernel_size=5,
    p_dim=3,
    in_channels=8,
    out_channels=16,
    KP_extent=1.0,
    radius=2.5,
    fixed_kernel_points="center",
    KP_influence="linear",
    aggregation_mode="sum",
    deformable=True,
    modulated=False,
)

In [195]:
set_seed(42)

# Example inputs:
q_pts = torch.rand(10, 3)  # 10 query points in 3D space
s_pts = torch.rand(15, 3)  # 15 support points in 3D space
neighb_inds = torch.randint(0, 15, (10, 5))  # Each query point has 5 neighbor indices
x = torch.rand(15, 8)  # 15 support points with 8 input channels/features

output = model(q_pts, s_pts, neighb_inds, x)

print("Output shape:", output.shape)

neighbors.shape = torch.Size([10, 5, 3]), deformed_K_points.shape = torch.Size([5, 3])
x.shape = torch.Size([16, 8]), new_neighb_inds.shape = torch.Size([10, 5])
neighbors.shape = torch.Size([10, 5, 3]), deformed_K_points.shape = torch.Size([10, 1, 5, 3])
x.shape = torch.Size([16, 8]), new_neighb_inds.shape = torch.Size([10, 5])
Output shape: torch.Size([10, 16])


In [196]:
def radius_gaussian(sq_r, sig, eps=1e-9):
    return torch.exp(-sq_r / (2 * sig**2 + eps))


def gather(x, idx, method=2):
    if method == 0:
        return x[idx]
    elif method == 1:
        x = x.unsqueeze(1)
        x = x.expand((-1, idx.shape[-1], -1))
        idx = idx.unsqueeze(2)
        idx = idx.expand((-1, -1, x.shape[-1]))
        return x.gather(0, idx)
    elif method == 2:
        for i, ni in enumerate(idx.size()[1:]):
            x = x.unsqueeze(i+1)
            new_s = list(x.size())
            new_s[i+1] = ni
            x = x.expand(new_s)
        n = len(idx.size())
        for i, di in enumerate(x.size()[n:]):
            idx = idx.unsqueeze(i+n)
            new_s = list(idx.size())
            new_s[i+n] = di
            idx = idx.expand(new_s)
        return x.gather(0, idx)
    else:
        raise ValueError('Unkown method')

In [131]:
import math
import torch
from torch import Tensor
from torch.nn.parameter import Parameter
from torch.nn.init import kaiming_uniform_


class KPConv_v2(nn.Module):
    def __init__(
        self,
        kernel_size: int,
        p_dim: int,
        in_channels: int,
        out_channels: int,
        KP_extent: float,
        radius: float,
        fixed_kernel_points: str = "center",
        KP_influence: str = "linear",
        aggregation_mode: str = "sum",
        deformable: bool = False,
        modulated: bool = False,
    ) -> None:
        super().__init__()

        self.kernel_size = kernel_size
        self.p_dim = p_dim
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.KP_extent = KP_extent
        self.radius = radius
        self.fixed_kernel_points = fixed_kernel_points
        self.KP_influence = KP_influence
        self.aggregation_mode = aggregation_mode
        self.deformable = deformable
        self.modulated = modulated

        # Initialize parameters
        self.weights = Parameter(torch.zeros(kernel_size, in_channels, out_channels), requires_grad=True)
        self.kernel_points = self._initialize_kernel_points()
        self.offset_conv, self.offset_bias = self._initialize_offsets() if deformable else (None, None)

        self.reset_parameters()

    def reset_parameters(self) -> None:
        """Reset the model parameters."""
        kaiming_uniform_(self.weights, a=math.sqrt(5))
        if self.offset_bias is not None:
            nn.init.zeros_(self.offset_bias)

    def _initialize_offsets(self) -> Tuple[nn.Module, Tensor]:
        """Initialize the offset convolution and bias."""
        offset_dim = (self.p_dim + 1) * self.kernel_size if self.modulated else self.p_dim * self.kernel_size
        offset_conv = KPConv(
            kernel_size=self.kernel_size,
            p_dim=self.p_dim,
            in_channels=self.in_channels,
            out_channels=offset_dim,
            KP_extent=self.KP_extent,
            radius=self.radius,
            fixed_kernel_points=self.fixed_kernel_points,
            KP_influence=self.KP_influence,
            aggregation_mode=self.aggregation_mode,
        )
        offset_bias = Parameter(torch.zeros(offset_dim), requires_grad=True)
        return offset_conv, offset_bias

    def _initialize_kernel_points(self) -> Tensor:
        """Load or create kernel points."""
        kernel_points = load_kernels(self.radius, self.kernel_size, dimension=self.p_dim, fixed=self.fixed_kernel_points)
        return Parameter(torch.tensor(kernel_points, dtype=torch.float32), requires_grad=False)

    def forward(self, q_pts: Tensor, s_pts: Tensor, neighb_inds: Tensor, x: Tensor) -> Tensor:
        """Forward pass of KPConv."""
        if self.deformable:
            self.offset_features = self.offset_conv(q_pts, s_pts, neighb_inds, x) + self.offset_bias
            offsets, modulations = self._compute_offsets_and_modulations()
        else:
            offsets, modulations = None, None

        return self._kpconv(q_pts, s_pts, neighb_inds, x, offsets, modulations)

    def _compute_offsets_and_modulations(self) -> Tuple[Tensor, Optional[Tensor]]:
        """Compute the offsets and modulations for deformable kernels."""
        unscaled_offsets = self.offset_features[:, :self.p_dim * self.kernel_size].view(-1, self.kernel_size, self.p_dim)
        offsets = unscaled_offsets * self.KP_extent
        modulations = 2 * torch.sigmoid(self.offset_features[:, self.p_dim * self.kernel_size:]) if self.modulated else None
        return offsets, modulations

    def _kpconv(
        self,
        q_pts: Tensor,
        s_pts: Tensor,
        neighb_inds: Tensor,
        x: Tensor,
        offsets: Optional[Tensor],
        modulations: Optional[Tensor],
    ) -> Tensor:
        """Main KPConv operation."""
        s_pts = torch.cat([s_pts, torch.zeros_like(s_pts[:1, :]) + 1e6], 0)
        neighbors = s_pts[neighb_inds, :] - q_pts.unsqueeze(1)

        if offsets is not None:
            deformed_K_points = self.kernel_points + offsets
        else:
            deformed_K_points = self.kernel_points

        sq_distances = torch.sum((neighbors.unsqueeze(2) - deformed_K_points) ** 2, dim=-1)
        all_weights = self._compute_weights(sq_distances)

        if self.aggregation_mode == "closest":
            all_weights = self._apply_closest_aggregation(sq_distances, all_weights)

        neighb_x = gather(x, neighb_inds)
        weighted_features = torch.matmul(all_weights, neighb_x)

        if self.deformable and modulations is not None:
            weighted_features *= modulations.unsqueeze(2)

        kernel_outputs = torch.matmul(weighted_features.permute(1, 0, 2), self.weights)
        return torch.sum(kernel_outputs, dim=0)

    def _compute_weights(self, sq_distances: Tensor) -> Tensor:
        """Compute kernel point weights based on influence type."""
        if self.KP_influence == "constant":
            return torch.ones_like(sq_distances).transpose(1, 2)
        elif self.KP_influence == "linear":
            return torch.clamp(1 - torch.sqrt(sq_distances) / self.KP_extent, min=0.0).transpose(1, 2)
        elif self.KP_influence == "gaussian":
            sigma = self.KP_extent * 0.3
            return radius_gaussian(sq_distances, sigma).transpose(1, 2)
        else:
            raise ValueError(f"Unknown influence type: {self.KP_influence}")

    def _apply_closest_aggregation(self, sq_distances: Tensor, all_weights: Tensor) -> Tensor:
        """Apply closest kernel point aggregation."""
        neighbors_1nn = torch.argmin(sq_distances, dim=2)
        return all_weights * torch.transpose(nn.functional.one_hot(neighbors_1nn, self.kernel_size), 1, 2)

    def extra_repr(self) -> str:
        return f"radius={self.radius}, in_channels={self.in_channels}, out_channels={self.out_channels}"

In [204]:
from torch_pointcloud.models.kpconv import KPConv as KPConv_v2

set_seed(42)

# Define the model
model_v2 = KPConv_v2(
    kernel_size=5,
    p_dim=3,
    in_channels=8,
    out_channels=16,
    KP_extent=1.0,
    radius=2.5,
    fixed_kernel_points="center",
    KP_influence="linear",
    aggregation_mode="sum",
    deformable=True,
    modulated=False,
)

In [205]:
# Make weights from model_v2 match model
model_v2.weights = model.weights
model_v2.kernel = model.kernel_points
if model_v2.deformable:
    model_v2.offset_conv = model.offset_conv
    model_v2.offset_bias = model.offset_bias

In [206]:
# neighbors.shape = torch.Size([10, 5, 3]), deformed_K_points.shape = torch.Size([5, 3])
# x.shape = torch.Size([16, 8]), new_neighb_inds.shape = torch.Size([10, 5])
# neighbors.shape = torch.Size([10, 5, 3]), deformed_K_points.shape = torch.Size([10, 1, 5, 3])
# x.shape = torch.Size([16, 8]), new_neighb_inds.shape = torch.Size([10, 5])
# Output shape: torch.Size([10, 16])

In [207]:
# Perform forward pass
output_v2 = model_v2(q_pts, s_pts, neighb_inds, x)

# Output shape: [n_points, out_channels]
print("Output shape:", output_v2.shape)
print("Output diff:", torch.norm(output - output_v2))
print(f"Same? {torch.allclose(output, output_v2)}")

neighbors.shape = torch.Size([10, 5, 3]), deformed_K_points.shape = torch.Size([5, 3])
x.shape = torch.Size([16, 8]), new_neighb_inds.shape = torch.Size([10, 5])
Output shape: torch.Size([10, 16])
Output diff: tensor(0., grad_fn=<LinalgVectorNormBackward0>)
Same? True


In [201]:
# x.shape = torch.Size([16, 8]), new_neighb_inds.shape = torch.Size([10, 5])
model.deformed_KP.shape

torch.Size([10, 5, 3])

In [203]:
model_v2

KPConv(radius=2.5, in_channels=8, out_channels=16 KP_extent=1.0, fixed_kernel_points=center, KP_influence=linear, aggregation_mode=sum, deformable=True, modulated=False)

## ResNet Block

In [280]:
class Config:
    KP_extent = 1.0
    conv_radius = 2.5
    use_batch_norm = True
    batch_norm_momentum = 0.1
    num_kernel_points = 15
    in_points_dim = 3
    fixed_kernel_points = "center"
    KP_influence = "linear"
    aggregation_mode = "closest"
    modulated = False
        
        
class DummyBatch:
    def __init__(self, n_points, n_neighbors, in_dim, n_layers):
        self.points = [torch.rand(n_points, 3) for _ in range(n_layers + 1)]  # Random 3D points
        self.neighbors = [torch.randint(0, n_points, (n_points, n_neighbors)) for _ in range(n_layers)]  # Neighbor indices
        self.pools = [torch.randint(0, n_points, (n_points, n_neighbors)) for _ in range(n_layers)]  # Pooling indices


# Instantiate the block with dummy config
config = Config()
n_points = 10
in_dim = 8
out_dim = 16
radius = 1.0
layer_ind = 0
n_neighbors = 5
n_layers = 3

# Create a dummy batch with random tensors
batch = DummyBatch(n_points, n_neighbors, in_dim, n_layers)

In [281]:
from models.blocks import ResnetBottleneckBlock, max_pool


set_seed(42)
block = ResnetBottleneckBlock(block_name="strided", in_dim=in_dim, out_dim=out_dim, radius=radius, layer_ind=layer_ind, config=config)

features = torch.rand(n_points, in_dim)  # Random features for n_points
output = block(features, batch)

[BatchNormBlock 1] x.shape = torch.Size([10, 4])
[BatchNormBlock 2] x.shape = torch.Size([10, 4, 1])
neighbors.shape = torch.Size([10, 5, 3]), deformed_K_points.shape = torch.Size([15, 3])
x.shape = torch.Size([11, 4]), new_neighb_inds.shape = torch.Size([10, 5])
[gather (x.gather(0, idx))] x.shape = torch.Size([11, 5, 4]), idx.shape = torch.Size([10, 5, 4])
[BatchNormBlock 1] x.shape = torch.Size([10, 4])
[BatchNormBlock 2] x.shape = torch.Size([10, 4, 1])
[BatchNormBlock 1] x.shape = torch.Size([10, 16])
[BatchNormBlock 2] x.shape = torch.Size([10, 16, 1])
[gather in] x.shape = torch.Size([11, 8]), inds.shape = torch.Size([10, 5])
[gather (x.gather(0, idx))] x.shape = torch.Size([11, 5, 8]), idx.shape = torch.Size([10, 5, 8])
[gather out] pool_features.shape = torch.Size([10, 5, 8])
[BatchNormBlock 1] x.shape = torch.Size([10, 16])
[BatchNormBlock 2] x.shape = torch.Size([10, 16, 1])


In [258]:
x = torch.rand(11, 8)  # Features for 11 points, each with 8 dimensions
idx = torch.randint(0, 11, (10, 5))  # Indices for 10 points, each with 5 neighbors

# Unsqueeze and expand inds to match the shape for gathering
# idxs = idxs.unsqueeze(-1).expand(-1, -1, x.size(1))
print(f"{x.shape = }")
print(f"{idx.shape = }")
# Use torch.gather to gather the corresponding features
torch.gather(x, 0, idx).shape

x.shape = torch.Size([11, 8])
idx.shape = torch.Size([10, 5])


torch.Size([10, 5])

In [259]:
for i, ni in enumerate(idx.size()[1:]):
    x = x.unsqueeze(i+1)
    new_s = list(x.size())
    new_s[i+1] = ni
    x = x.expand(new_s)

n = len(idx.size())
for i, di in enumerate(x.size()[n:]):
    idx = idx.unsqueeze(i+n)
    new_s = list(idx.size())
    new_s[i+n] = di
    idx = idx.expand(new_s)
    
# indices.unsqueeze(-1).repeat(1, 1, D)

In [262]:
nn.BatchNorm1d(8, momentum=0.1).bias

Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0.], requires_grad=True)

In [274]:
class BatchNormBlock(nn.Module):
    def __init__(self, in_dim, use_bn, bn_momentum):
        super().__init__()
        self.batch_norm = nn.BatchNorm1d(in_dim, momentum=bn_momentum)

    def forward(self, x):
        print(f"[bn 1] {x.shape = }")
        x = x.unsqueeze(2)
        x = x.transpose(0, 2)
        print(f"[bn 2] {x.shape = }")
        x = self.batch_norm(x)
        x = x.transpose(0, 2)
        return x.squeeze()
        
        
class UnaryBlock(nn.Module):
    def __init__(self, in_dim, out_dim, use_bn, bn_momentum, no_relu=False):
        super().__init__()
        self.bn_momentum = bn_momentum
        self.use_bn = use_bn
        self.no_relu = no_relu
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.mlp = nn.Linear(in_dim, out_dim, bias=False)
        self.batch_norm = BatchNormBlock(out_dim, self.use_bn, self.bn_momentum)
        if not no_relu:
            self.leaky_relu = nn.LeakyReLU(0.1)
        return

    def forward(self, x, batch=None):
        x = self.mlp(x)
        x = self.batch_norm(x)
        if not self.no_relu:
            x = self.leaky_relu(x)
        return x

In [275]:
class ResnetBottleneckBlock(nn.Module):
    def __init__(
        self,
        block_name: str,
        in_dim: int,
        out_dim: int,
        radius: float,
        layer_ind: int,
        config,
    ) -> None:
        super().__init__()

        # Calculate KP extent from current radius
        current_extent = radius * config.KP_extent / config.conv_radius

        # Save configuration options
        self.use_bn = config.use_batch_norm
        self.bn_momentum = config.batch_norm_momentum
        self.layer_ind = layer_ind
        self.block_name = block_name

        # Downscaling block
        self.unary1 = UnaryBlock(in_dim, out_dim // 4, self.use_bn, self.bn_momentum)

        # KPConv block with batch normalization
        self.conv = KPConv(
            config.num_kernel_points,
            config.in_points_dim,
            out_dim // 4,
            out_dim // 4,
            current_extent,
            radius,
            fixed_kernel_points=config.fixed_kernel_points,
            KP_influence=config.KP_influence,
            aggregation_mode=config.aggregation_mode,
            deformable='deform' in block_name,
            modulated=config.modulated
        )
        self.bn = BatchNormBlock(out_dim // 4, self.use_bn, self.bn_momentum)
        # self.bn = nn.BatchNorm1d(out_dim // 4, momentum=self.bn_momentum)

        # Upscaling block
        self.unary2 = UnaryBlock(out_dim // 4, out_dim, self.use_bn, self.bn_momentum, no_relu=True)

        # Shortcut block
        self.unary_shortcut = UnaryBlock(in_dim, out_dim, self.use_bn, self.bn_momentum, no_relu=True)

        # Activation function
        self.leaky_relu = nn.LeakyReLU(0.1)

    def forward(self, features: Tensor, batch) -> Tensor:
        """Forward pass of the ResNet bottleneck block.

        Args:
            features (Tensor): Input feature tensor.
            batch: Batch object containing points, neighbors, and pools.

        Returns:
            Tensor: Output feature tensor after the bottleneck block.
        """
        q_pts, s_pts, neighb_inds = self._get_points_and_neighbors(batch)

        # Apply first downscaling MLP
        x = self.unary1(features)

        # Apply KPConv
        x = self.conv(q_pts, s_pts, neighb_inds, x)

        # x = x.unsqueeze(2).transpose(0, 2)
        x = self.leaky_relu(self.bn(x))
        # x = x.transpose(0, 2).squeeze()

        # Apply second upscaling MLP
        x = self.unary2(x)

        # Apply shortcut and combine
        shortcut = self._get_shortcut(features, neighb_inds)
        return self.leaky_relu(x + shortcut)

    def _get_points_and_neighbors(self, batch) -> Tuple[Tensor, Tensor, Tensor]:
        """Helper function to get points and neighbors based on the block type."""
        if 'strided' in self.block_name:
            q_pts = batch.points[self.layer_ind + 1]
            s_pts = batch.points[self.layer_ind]
            neighb_inds = batch.pools[self.layer_ind]
        else:
            q_pts = batch.points[self.layer_ind]
            s_pts = batch.points[self.layer_ind]
            neighb_inds = batch.neighbors[self.layer_ind]
        return q_pts, s_pts, neighb_inds

    def _get_shortcut(self, features: Tensor, neighb_inds: Tensor) -> Tensor:
        if 'strided' in self.block_name:
            return self.unary_shortcut(max_pool(features, neighb_inds))
        else:
            return self.unary_shortcut(features)

In [276]:
class Config:
    KP_extent = 1.0
    conv_radius = 2.5
    use_batch_norm = True
    batch_norm_momentum = 0.1
    num_kernel_points = 15
    in_points_dim = 3
    fixed_kernel_points = "center"
    KP_influence = "linear"
    aggregation_mode = "closest"
    modulated = False
        
        
class DummyBatch:
    def __init__(self, n_points, n_neighbors, in_dim, n_layers):
        self.points = [torch.rand(n_points, 3) for _ in range(n_layers + 1)]  # Random 3D points
        self.neighbors = [torch.randint(0, n_points, (n_points, n_neighbors)) for _ in range(n_layers)]  # Neighbor indices
        self.pools = [torch.randint(0, n_points, (n_points, n_neighbors)) for _ in range(n_layers)]  # Pooling indices


# Instantiate the block with dummy config
config = Config()
n_points = 10
in_dim = 8
out_dim = 16
radius = 1.0
layer_ind = 0
n_neighbors = 5
n_layers = 3

# Create a dummy batch with random tensors
batch = DummyBatch(n_points, n_neighbors, in_dim, n_layers)

In [277]:
set_seed(42)
block = ResnetBottleneckBlock(block_name="strided", in_dim=in_dim, out_dim=out_dim, radius=radius, layer_ind=layer_ind, config=config)

features = torch.rand(n_points, in_dim)  # Random features for n_points
output = block(features, batch)

[bn 1] x.shape = torch.Size([10, 4])
[bn 2] x.shape = torch.Size([1, 4, 10])
neighbors.shape = torch.Size([10, 5, 3]), deformed_K_points.shape = torch.Size([15, 3])
x.shape = torch.Size([11, 4]), new_neighb_inds.shape = torch.Size([10, 5])
[gather (x.gather(0, idx))] x.shape = torch.Size([11, 5, 4]), idx.shape = torch.Size([10, 5, 4])
[bn 1] x.shape = torch.Size([10, 4])
[bn 2] x.shape = torch.Size([1, 4, 10])
[bn 1] x.shape = torch.Size([10, 16])
[bn 2] x.shape = torch.Size([1, 16, 10])
[gather in] x.shape = torch.Size([11, 8]), inds.shape = torch.Size([10, 5])
[gather (x.gather(0, idx))] x.shape = torch.Size([11, 5, 8]), idx.shape = torch.Size([10, 5, 8])
[gather out] pool_features.shape = torch.Size([10, 5, 8])
[bn 1] x.shape = torch.Size([10, 16])
[bn 2] x.shape = torch.Size([1, 16, 10])
